In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Download list of Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).rename({'column_1': 'region'})
olink_genes

Error: path
"/home/dnanexus/ukbgym/utils/average_pheno_per_variant/proteomics_genes.txt"
already exists but -f/--overwrite was not set


region
str
"""ENSG00000266967"""
"""ENSG00000114779"""
"""ENSG00000097007"""
"""ENSG00000060971"""
"""ENSG00000157766"""
…
"""ENSG00000173465"""
"""ENSG00000105428"""
"""ENSG00000188372"""


In [3]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym_with_mane.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet', 
    columns=['id', 'region']
)

anno

Error: path
"/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


id,region
str,str
"""chr8:134702216:A:G""","""ENSG00000066827"""
"""chr19:47846298:C:T""","""ENSG00000105392"""
"""chr17:49763088:G:A""","""ENSG00000121104"""
"""chr8:38172782:G:C""","""ENSG00000175324"""
"""chr1:174771515:C:T""","""ENSG00000152061"""
…,…
"""chr11:70472380:A:T""","""ENSG00000162105"""
"""chr8:16174283:C:T""","""ENSG00000038945"""
"""chr2:215381006:G:C""","""ENSG00000115414"""


In [4]:
anno['region'].value_counts(sort=True).join(olink_genes, on='region', how='semi')

region,count
str,u64
"""ENSG00000174469""",605106
"""ENSG00000185008""",444475
"""ENSG00000189283""",435466
"""ENSG00000021645""",385076
"""ENSG00000149972""",376497
…,…
"""ENSG00000179889""",1616
"""ENSG00000160221""",669
"""ENSG00000267368""",611


In [5]:
# prot_file = "cauc_cov_regression_90pcs_prs"  # covariates and PRS corrected
# prot_file = "cauc_protrider_lite_prs_rint"   # PROTRIDER corrected (RINT)
prot_file = "cauc_protrider_lite_prs_t_df"     # PROTRIDER corrected (T distribution)

# Download Olink:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/{prot_file}.parquet -o /home/dnanexus/data_dir/

phenos = pl.read_parquet(f'/home/dnanexus/data_dir/{prot_file}.parquet')

olink_genes_w_data = list(set(phenos.columns).intersection(set(olink_genes['region'])).intersection(set(anno['region'])))

phenos = phenos.select(['sample'] + olink_genes_w_data)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes_w_data,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        region = pl.col('phenotype'),
        phenotype = pl.col('phenotype') + '_olink',
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path "/home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df.parquet"
already exists but -f/--overwrite was not set
shape: (2_661, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000007908_olink ┆ 39208 │
│ ENSG00000275718_olink ┆ 39208 │
│ ENSG00000095713_olink ┆ 39208 │
│ ENSG00000275152_olink ┆ 39208 │
│ ENSG00000106991_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000102837_olink ┆ 31597 │
│ ENSG00000131050_olink ┆ 31502 │
│ ENSG00000111405_olink ┆ 30953 │
│ ENSG00000163131_olink ┆ 30432 │
│ ENSG00000170373_olink ┆ 29106 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value,region
str,str,f64,str
"""5645319""","""ENSG00000168488_olink""",0.196207,"""ENSG00000168488"""
"""5959139""","""ENSG00000168488_olink""",1.152436,"""ENSG00000168488"""
"""5732867""","""ENSG00000168488_olink""",0.369172,"""ENSG00000168488"""
"""2074480""","""ENSG00000168488_olink""",2.643769,"""ENSG00000168488"""
"""4659532""","""ENSG00000168488_olink""",0.225678,"""ENSG00000168488"""
…,…,…,…
"""1807196""","""ENSG00000229314_olink""",1.47917,"""ENSG00000229314"""
"""4223555""","""ENSG00000229314_olink""",-0.961273,"""ENSG00000229314"""
"""2992773""","""ENSG00000229314_olink""",1.014887,"""ENSG00000229314"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)
)
long_gt.head().collect()

[===========================================================>] Completed 27,974,638,284 of 27,974,638,284 bytes (100%) /home/dnanexus/data_dir/gt_long.parquett====>                                                      ] Downloaded 2,919,235,584 of 27,974,638,284 bytes (10%) /home/dnanexus/data_dir/gt_long.parquet[======>                                                     ] Downloaded 3,120,562,176 of 27,974,638,284 bytes (11%) /home/dnanexus/data_dir/gt_long.parquet


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [7]:
output_dir = '/home/dnanexus/data_dir/appv_phenos/'
!mkdir -p {output_dir}

CHUNK_SIZE = 100
total_genes = len(olink_genes_w_data)
num_chunks = math.ceil(total_genes / CHUNK_SIZE)

print(f"Processing {total_genes} genes in {num_chunks} chunks...")

# Process in Batches
for i in tqdm(range(0, total_genes, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_genes = olink_genes_w_data[i : i + CHUNK_SIZE]
    chunk_phenos = [f"{g}_olink" for g in chunk_genes]

# for olink_gene in tqdm(olink_genes_w_data):
    print(f"Processing chunk starting at index: {i}")
    
    (
        anno.filter(pl.col('region').is_in(chunk_genes))
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype').is_in(chunk_phenos)).lazy(),
            on=['sample', 'region'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(descending=True, method="max")
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk{i}.parquet')
        # .collect(engine='streaming')
    )

Processing 2661 genes in 27 chunks...


  0%|          | 0/27 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  4%|▎         | 1/27 [00:14<06:07, 14.13s/it]

Processing chunk starting at index: 100


...

 93%|█████████▎| 25/27 [05:44<00:27, 13.71s/it]

Processing chunk starting at index: 2500


 96%|█████████▋| 26/27 [05:59<00:13, 13.86s/it]

Processing chunk starting at index: 2600


100%|██████████| 27/27 [06:12<00:00, 13.79s/it]


## Consolidate parquet

In [8]:
pl.read_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk0.parquet')

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:51619350:A:G""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.861672,null,544058.0,0.860699
"""chr10:51498895:T:C""","""ENSG00000185532""","""ENSG00000185532_olink""",2,-0.652455,1.052694,508152.0,0.803896
"""chr10:51595746:C:A""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.838022,null,540359.0,0.854847
"""chr10:51541835:TTAAC:T""","""ENSG00000185532""","""ENSG00000185532_olink""",2,-0.054503,0.088364,337718.0,0.534269
"""chr10:51592133:G:A""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.307818,null,422965.0,0.66913
…,…,…,…,…,…,…,…
"""chr10:52198073:A:G""","""ENSG00000185532""","""ENSG00000185532_olink""",7,-0.365915,1.331381,439886.0,0.695899
"""chr10:52196888:A:T""","""ENSG00000185532""","""ENSG00000185532_olink""",1,0.891976,null,83832.0,0.132622
"""chr10:52196762:A:G""","""ENSG00000185532""","""ENSG00000185532_olink""",2,-0.283181,0.835938,415488.0,0.657301


In [9]:
combined_output_file = f"/home/dnanexus/data_dir/{prot_file}_all_genes_EURunrelated_appv_percentiles_descending.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...
Done.


In [10]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 313,366,309 of 313,366,309 bytes (100%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles_descending.parquet
ID                                file-J60qJ6QJg0y2JG35q7XY1GGf
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/avg_pheno_per_var
Name                              cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentil
                                  es_descending.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Tue Feb  3 08:49:30 2026
Created by                        shubhankar
 via the job                      job-J60q

In [11]:
a = pl.scan_parquet(combined_output_file)
a.head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:51619350:A:G""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.861672,null,544058.0,0.860699
"""chr10:51498895:T:C""","""ENSG00000185532""","""ENSG00000185532_olink""",2,-0.652455,1.052694,508152.0,0.803896
"""chr10:51595746:C:A""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.838022,null,540359.0,0.854847
"""chr10:51541835:TTAAC:T""","""ENSG00000185532""","""ENSG00000185532_olink""",2,-0.054503,0.088364,337718.0,0.534269
"""chr10:51592133:G:A""","""ENSG00000185532""","""ENSG00000185532_olink""",1,-0.307818,null,422965.0,0.66913


In [12]:
a.select(['region']).collect()['region'].unique()

region
str
"""ENSG00000169900"""
"""ENSG00000150995"""
"""ENSG00000106992"""
"""ENSG00000119630"""
"""ENSG00000104450"""
…
"""ENSG00000204539"""
"""ENSG00000113555"""
"""ENSG00000169242"""
